# 05 - Does patching the model's GPRs blur the ECS embedding?

The audit against Mouse-GEM (Mouse1) found genes missing from the GPRs of reactions
mitoMAMMALmod already contains. Adding them is only free if it costs nothing downstream, and
that is not obvious: `split_isozymes` gives every OR branch its own ECS column, so an isozyme
that is not expressed still becomes a feature carrying only the ambient floor. More noise
dimensions for PCA and the neighbour graph to absorb means a blobbier UMAP, and blobbier is
worse.

This notebook runs `cleaning_report` -> `calculate_ecs` -> `module_1` once per model variant
and measures whether cell types still separate.

| variant | what it is |
|---|---|
| `baseline` | mitoMAMMALmod as it ships |
| `repaired` | bug fixes only, no genes added - **the control** |
| `conservative` | complex subunits and MitoCarta-backed isozymes |
| `inclusive` | everything the tissue-independent filters allow |

`repaired` is the honest control: it fixes GPRs that were silently scoring zero (unparsable
`GENE_ASSOCIATION` on `OIVD2m` and `r1451`, `GluForTx` keeping only the human ortholog, a
mistyped Ensembl id for Ttc19, Nnt's id) without adding any genes, so the gap between
`repaired` and the other two is the effect of the added genes alone.

Both OR strategies are run. Under `max` an unexpressed isozyme genuinely cannot move a score;
under `sum` - the `ecs_calculator` default - it accumulates ambient signal. If the two runs
disagree, that is an argument for changing the default rather than for trimming the model.


## 0 - Settings

In [ ]:
import os, sys, glob

BASE = '/scratch/prj/crb_inner_ear/k2147692/metabolic'
REPO_DIR = f'{BASE}/code/Metabolic-pipeline'
DATA_PATH = f'{BASE}/data/kolla/kolla_E16.h5ad'
OUT_SUM = f'{BASE}/results/05_variant_comparison_sum'
OUT_MAX = f'{BASE}/results/05_variant_comparison_max'

CELLTYPE_COL = 'cell_type'
SYMBOL_COL = 'gene_symbol'
AND_STRATEGY = 'median'
RESOLUTION = 1.0
RESUME = True          # score variants already on disk instead of recomputing them

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print('repo:', REPO_DIR)
print('data:', DATA_PATH, '|', 'found' if os.path.exists(DATA_PATH) else 'MISSING')


## 1 - Check what this session actually has

`calculate_ecs` densifies the expression matrix and then builds a per-gene dictionary from it,
costing roughly two dense `float32` copies of the matrix, once per variant.

If this is running on a **login node**, or in a session with a small memory limit, that is what
makes it crawl - it will be swapping rather than computing. A CREATE compute node is named
something like `erc-hpc-comp###`, a login node `erc-hpc-login###`.

If you are on a login node, or `mem_limit` below is under about 16 GB, close this session and
start a new OOD Jupyter session asking for **4 cores and 32 GB**. Nothing else here changes.


In [ ]:
import socket

print('hostname :', socket.gethostname())
print('cpus     :', len(os.sched_getaffinity(0)) if hasattr(os, 'sched_getaffinity') else os.cpu_count())
for var in ('SLURM_JOB_ID', 'SLURM_CPUS_ON_NODE', 'SLURM_MEM_PER_NODE'):
    print(f'{var:18s}', os.environ.get(var, '(not set - not inside a Slurm job)'))

# the cgroup limit is what actually kills the process, not the node's total RAM
for path in ('/sys/fs/cgroup/memory.max', '/sys/fs/cgroup/memory/memory.limit_in_bytes'):
    if os.path.exists(path):
        raw = open(path).read().strip()
        print('mem_limit:', f'{int(raw) / 2**30:.1f} GB' if raw.isdigit() else raw)
        break

import anndata
a = anndata.read_h5ad(DATA_PATH, backed='r')
dense_gb = a.n_obs * a.n_vars * 4 / 2**30
print(f'\ndata: {a.n_obs} cells x {a.n_vars} genes -> {dense_gb:.2f} GB dense, '
      f'so expect roughly {2 * dense_gb:.1f} GB peak per variant')
print('obs columns:', list(a.obs.columns))
assert CELLTYPE_COL in a.obs.columns, f'{CELLTYPE_COL!r} not in obs; pick one of the above'
del a


## 2 - Run the comparison

Four variants per strategy, each running the full chain and writing a complete module_1 output
folder (UMAP PDFs, residual heatmap, annotated h5ad).

With `RESUME = True` a variant whose `Annotated_RCS.h5ad` already exists is scored from disk
rather than recomputed, so if the session times out you can rerun the cell and it carries on.


In [ ]:
from metabolic_tools.model_variant_benchmark import compare_variants

summary_sum = compare_variants(
    DATA_PATH, OUT_SUM, resume=RESUME,
    celltype_col=CELLTYPE_COL, gene_column=SYMBOL_COL,
    and_strategy=AND_STRATEGY, or_strategy='sum', cluster_resolution=RESOLUTION)


In [ ]:
summary_max = compare_variants(
    DATA_PATH, OUT_MAX, resume=RESUME,
    celltype_col=CELLTYPE_COL, gene_column=SYMBOL_COL,
    and_strategy=AND_STRATEGY, or_strategy='max', cluster_resolution=RESOLUTION)


## 3 - Read the result

`silhouette_pca` and `knn_purity` are the two to trust: PCA and the neighbour graph are
deterministic here, so a change in those is a real consequence of the feature set rather than a
different random seed. `ari`, `nmi` and `umap_separation` depend on seeds scanpy fixes at 0 -
reproducible, but a single draw.

The failure mode to look for is `n_features` rising while the separation metrics fall. If they
hold or improve, the extra isozymes cost nothing and the patch is safe to publish.


In [ ]:
import pandas as pd

pd.set_option('display.width', 200)
metrics = ['n_features', 'silhouette_pca', 'knn_purity', 'ari', 'nmi',
           'umap_separation', 'mean_abs_residual', 'n_clusters']
both = pd.concat({'sum': summary_sum[metrics], 'max': summary_max[metrics]}, axis=0)
display(both.round(4))

print('\nchange vs baseline:')
for label, table in (('sum', summary_sum), ('max', summary_max)):
    delta = table[metrics] - table[metrics].loc['baseline']
    print(f'\n-- or_strategy={label}')
    print(delta.drop(index='baseline').round(4).to_string())


In [ ]:
import matplotlib.pyplot as plt

show = ['silhouette_pca', 'knn_purity', 'nmi', 'umap_separation']
order = [v for v in ['baseline', 'repaired', 'conservative', 'inclusive'] if v in summary_sum.index]
fig, axes = plt.subplots(1, len(show), figsize=(4 * len(show), 3.4))
for ax, metric in zip(axes, show):
    for label, table, marker in (('sum', summary_sum, 'o'), ('max', summary_max, 's')):
        ax.plot(order, table.loc[order, metric], marker=marker, label=f'or={label}')
    ax.set_title(metric)
    ax.tick_params(axis='x', rotation=45)
    ax.spines[['top', 'right']].set_visible(False)
axes[0].set_ylabel('higher = cell types separate better')
axes[-1].legend(frameon=False)
plt.tight_layout()
plt.savefig(f'{OUT_SUM}/variant_metrics.pdf', bbox_inches='tight')
plt.show()


The four UMAPs are worth looking at directly, one per variant, in
`<out_dir>/<variant>/UMAP_Metabolic_Archetypes.pdf` and `UMAP_Original_CellTypes.pdf`. The
metrics say whether structure was lost; the plots say where.


In [ ]:
for out_dir in (OUT_SUM, OUT_MAX):
    print(out_dir)
    for pdf in sorted(glob.glob(f'{out_dir}/*/UMAP_Original_CellTypes.pdf')):
        print('   ', pdf)


## 4 - Sending results back

Zip both result directories, then pull the archive down with `scp` from **PowerShell on the
laptop** - the command is printed below. The `-o MACs=hmac-sha2-512` is required or the
connection will not negotiate.


In [ ]:
import shutil

staging = f'{BASE}/results/05_variant_comparison'
os.makedirs(staging, exist_ok=True)
for src in (OUT_SUM, OUT_MAX):
    dst = os.path.join(staging, os.path.basename(src))
    if os.path.exists(dst):
        shutil.rmtree(dst)
    shutil.copytree(src, dst, ignore=shutil.ignore_patterns('*.h5ad'))
archive = shutil.make_archive(staging, 'zip', root_dir=staging)
print('wrote', archive, f'({os.path.getsize(archive) / 2**20:.0f} MB, h5ads excluded)')

print('\nRun this in PowerShell on the laptop:\n')
print('cd "C:\\Users\\James\\OneDrive - King' + "'" + 's College London\\Documents\\Metabolic-results"')
print(f'scp -o MACs=hmac-sha2-512 k2147692@hpc.create.kcl.ac.uk:{archive} .')
print('Expand-Archive 05_variant_comparison.zip -DestinationPath .')
